**The information in this table is related to housing prices in the country of Australia and the state of the Northern Territory.**

| Column name      | Description                                                                   |
|------------------|-------------------------------------------------------------------------------|
| breadcrumb       | A breadcrumb is a text trail that shows the user's location within a website. (String) |
| category_name    | The name of the category that the listing belongs to. (String)                   |
| property_type    | The type of property being listed. (String)                                     |
| building_size    | The size of the property's building, in square meters. (Numeric)                |
| land_size        | The size of the property's land, in square meters. (Numeric)                    |
| preferred_size   | The preferred size of the property, in square meters. (Numeric)                 |
| open_date        | The date that the property was first listed for sale. (Date)                    |
| listing_agency   | The agency that is listing the property. (String)                               |
| price            | The listing price of the property. (Numeric)                                    |
| location_number  | The number that corresponds to the property's location. (Numeric)               |
| location_type    | The type of location that the property is in. (String)                          |
| location_name    | The name of the location that the property is in. (String)                      |
| address          | The property's address. (String)                                                |
| address_1        | The first line of the property's address. (String)                              |
| city             | The city that the property is located in. (String)                              |
| state            | The state that the property is located in. (String)                             |
| zip_code         | The zip code that the property is located in. (String)                          |
| phone            | The listing agent's phone number. (String)                                      |
| latitude         | The property's latitude. (Numeric)                                              |
| longitude        | The property's longitude. (Numeric)                                              |
| product_depth    | The depth of the product. (Numeric)                                              |
| bedroom_count    | The number of bedrooms in the property. (Numeric)                                |
| bathroom_count   | The number of bathrooms in the property. (Numeric)                               |
| parking_count    | The number of parking spaces in the property. (Numeric)                         |
| RunDate          | The date that the listing was last updated. (Date)                              |


In [ ]:
import pandas as pd
pd.set_option('display.float_format', '{:.2f}'.format)
import numpy as np
np.set_printoptions(precision=15, floatmode='fixed')
import seaborn as sns
import matplotlib.pyplot as plt

import re
from sklearn.preprocessing import LabelEncoder

from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn import metrics
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings("ignore")

In [ ]:
data = pd.read_csv('RealEstateAU_1000_Samples.csv')
df = pd.DataFrame(data)
df

# Data cleaning

In [ ]:
df.isna().sum()

*We ran the code df.isna().sum() to check the number of missing values (NaN) in each column of the dataset. This is an essential step in the data cleaning process, as it helps us identify the columns that may require special attention or treatment.*

In [ ]:
df = df.drop(columns=['building_size','land_size','preferred_size','open_date','latitude','longitude'])
df.isna().sum()

We ran the code to drop the columns `building_size`, `land_size`, `preferred_size`, `open_date`, `latitude`, and `longitude` from the dataframe.

1. **Building Size, Land Size, and Preferred Size:** These columns had a substantial number of missing values, and if they are not essential to the analysis, dropping them simplifies the data handling.
2. **Open Date:** This had a significant number of missing values and was dropped for similar reasons.
3. **Latitude and Longitude:** Since all the values were missing in these columns, they were removed as they provide no information.


In [ ]:

# Finding pairs of columns with more than 90% similarity
duplicate_columns = []
threshold = 0.9 * len(df)

for col1 in df.columns:
    for col2 in df.columns:
        if col1 != col2 and (col2, col1) not in duplicate_columns:
            similarity_count = (df[col1] == df[col2]).sum()
            if similarity_count > threshold:
                duplicate_columns.append((col1, col2))

duplicate_columns


Identifying columns that are highly similar helps in detecting redundancy, which can be a sign of data duplication or unnecessary repetition and Removing redundant columns can reduce the complexity of the dataset, making subsequent analyses more efficient and interpretable.

### Interpretation of Results:
The result indicates that the columns `price` and `location_name` have more than 90% similarity. This finding may be surprising, as price and location name are not typically expected to be similar. It might be worth investigating this further to understand the relationship between these columns and decide whether one of them should be removed or whether there's an underlying issue in the data.

In [ ]:
df = df.drop(columns=['location_name'])

As identified earlier, the location_name column had more than 90% similarity with the price column. Dropping it helps eliminate redundancy in the dataset.

In [ ]:
# Checking if all values in each column are the same
columns_with_same_values = []

for col in df.columns:
    first_value = df[col].iloc[0]
    if df[col].eq(first_value).all():
        columns_with_same_values.append(col)

columns_with_same_values


Constant columns (i.e., columns where all values are the same) don't contribute to statistical analyses or models, as they don't provide any differentiation between observations.

In [ ]:
df = df.drop(columns=['location_type','state','RunDate'])

In [ ]:
df = df.drop(columns=['index','TID','breadcrumb','category_name','address','address_1','phone'])

*The remaining columns are those whose information is directly given in other columns or are not worth checking, such as an index*

In [ ]:
df.isna().sum()

In [ ]:
df = df.dropna()
df.isna().sum()

In [ ]:
df

In [ ]:
# function to extract price from the text
def extract_price(price_text):
    # Using regex to find a pattern that matches the price and ignores any following text
    price_match = re.search(r'\$[\d, ]*(?:\.\d{2})?', str(price_text))
    if price_match:
        # Remove spaces from the captured price and return
        return price_match.group(0).replace(' ', '').replace(',', '')
    else:
        return None

# Applying the improved function to the "price" column
df['price'] = df['price'].apply(extract_price)

df.head(3)



The `price` column may contain textual information or additional characters. The function uses a regular expression (regex) to extract the numeric price value, including the currency symbol.By applying this function to the `price` column, we transform the prices into a consistent format that can be used for numerical analysis.


In [ ]:
df.isna().sum()

In [ ]:
df = df.dropna()
df.isna().sum()

In [ ]:
df_copy = df.copy()
df_copy['price'] = df_copy['price'].replace('[\$,.]', '', regex=True).astype(int)
df = df_copy
df= df.reset_index(drop=True)
df

By removing currency symbols and commas, and converting the `price` column to an integer type, you have ensured that this column is now in a numerical format suitable for statistical analysis and modeling.

In [ ]:
#The function check_dtypes inspects the unique data types in each column of the dataframe.
def check_dtypes(df):
    dtypes = []
    for col in df.columns:
        dtypes.append( (col, df[col].apply(type).unique()) ) 
    return pd.DataFrame(dtypes, columns=['Column Name', 'Data Type'])


In [ ]:
check_dtypes(df)

In [ ]:
negative_count = df[['price','location_number','zip_code','bedroom_count','bathroom_count','parking_count']].apply(lambda x: (x < 0).sum())
negative_count

checks for negative values in the specified numeric columns: `'price'`, `'location_number'`, `'zip_code'`, `'bedroom_count'`, `'bathroom_count'`, and `'parking_count'`.

Negative values in these columns might not make logical sense (e.g., negative price or negative bedroom count), so checking for them helps identify potential errors or inconsistencies in the data.

# Noise Detecting

In [ ]:
plot_cols = df.columns[ df.columns.isin(['price','zip_code','bedroom_count','bathroom_count','parking_count']) ]

# Calculate the number of rows for subplots
n_cols = 3
n_rows = -(-len(plot_cols) // n_cols)  # ceil division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 5), squeeze=False)

for i, col in enumerate(plot_cols):
    row, col_idx = divmod(i, n_cols)
    sns.scatterplot(data=df, x=df.index, y=col, edgecolor="0", ax=axes[row, col_idx])
    axes[row, col_idx].set_title(f'Scatter plot of {col} vs Index')
    axes[row, col_idx].set_xlabel('Index')
    axes[row, col_idx].set_ylabel(col)

# Remove empty subplots
for j in range(i+1, n_rows*n_cols):
    row, col_idx = divmod(j, n_cols)
    fig.delaxes(axes[row][col_idx])

plt.tight_layout()
plt.show()

The scatter plots you requested have been created for the columns `'price'`, `'zip_code'`, `'bedroom_count'`, `'bathroom_count'`, and `'parking_count'`. Each plot shows the respective column's values plotted against the index.

### Interpretation of Results:
- **Price:** Seems to have a varied distribution, with most properties falling in a specific range, but there are some higher-priced outliers.
- **Zip Code:** The distribution appears consistent, with no noticeable outliers.
- **Bedroom Count, Bathroom Count, Parking Count:** These features show discrete values, as expected, with no apparent anomalies.

In [ ]:
df['price'].nsmallest(10)

In [ ]:
df = df[ df['price'] < 3990000 ]
df = df[ df['price'] > 800 ]
df = df.reset_index(drop=True)

By applying this filter, we removing outlier properties that have unusually high prices. This can help ensure that the analysis focuses on a more typical range of properties.

In [ ]:
plot_cols = df.columns[ df.columns.isin(['price','zip_code','bedroom_count','bathroom_count','parking_count']) ]

# Calculate the number of rows for subplots
n_cols = 3
n_rows = -(-len(plot_cols) // n_cols)  # ceil division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 5), squeeze=False)

for i, col in enumerate(plot_cols):
    row, col_idx = divmod(i, n_cols)
    sns.scatterplot(data=df, x=df.index, y=col, edgecolor="0", ax=axes[row, col_idx])
    axes[row, col_idx].set_title(f'Scatter plot of {col} vs Index')
    axes[row, col_idx].set_xlabel('Index')
    axes[row, col_idx].set_ylabel(col)

# Remove empty subplots
for j in range(i+1, n_rows*n_cols):
    row, col_idx = divmod(j, n_cols)
    fig.delaxes(axes[row][col_idx])

plt.tight_layout()
plt.show()

# EDA

In [ ]:
df.describe()

Certainly! Let's delve deeper into the descriptive statistics provided for the dataset:

### 1. Price:
- **Range:** The minimum price is \$99950 , and the maximum is \$1,950,000. This wide range indicates significant diversity in the property prices.
- **Mean:** The average price is approximately \$518,588.7, suggesting a central tendency towards this value.
- **Quartiles:** 50% of the properties are priced between \$380,000 (25th percentile) and \$610,000 (75th percentile).
- **Standard Deviation:** A standard deviation of \$230,647.5 indicates the spread or variability in the prices.

### 2. Location Number:
- **Range and Mean:** These values may correspond to specific location codes or identifiers. A deeper understanding of what this number represents would provide more insights.

### 3. Zip Code:
- **Range:** Zip codes range from 800 to 839, indicating the geographic coverage of the dataset.
- **Mean:** The mean zip code value may not provide significant insights, as zip codes are categorical rather than continuous variables.

### 4. Bedroom Count:
- **Range:** From 0 to 8 bedrooms, indicating a variety of property sizes.
- **Mean:** The average number of bedrooms is approximately 2.77.
- **Standard Deviation:** A standard deviation of 1.09 suggests moderate variability in bedroom counts.

### 5. Bathroom Count:
- **Range:** 1 to 5 bathrooms, indicating diversity in amenities.
- **Mean:** The average number of bathrooms is around 1.70.
- **Standard Deviation:** A standard deviation of 0.61 suggests most properties have 1 or 2 bathrooms.

### 6. Parking Count:
- **Range:** From 0 to 12 parking spaces, highlighting varied parking facilities.
- **Mean:** The average number of parking spaces is approximately 2.10.
- **Standard Deviation:** A standard deviation of 1.51 indicates a wider spread in parking availability.


In [ ]:
plt.figure(figsize=(35,10), dpi=200)
ax =sns.countplot(x='property_type' , data= df )
plt.xlabel('property_type',fontsize=30)
plt.xticks(rotation = 90 , fontsize=30)
plt.ylabel('count',fontsize=30)
plt.yticks(fontsize=20)
plt.title('property_type' , fontsize=30)
plt.grid()
plt.show()

Here's a summary of what we observe:

- **House**: The most common property type, with a significant number of listings.
- **Unit**: The second most common property type, also with a substantial number of listings.
- **Apartment**: Fewer listings than houses and units, but still a noticeable presence.
- **Other Types**: A few other property types, such as "Land" and "Townhouse," have smaller counts in the dataset.

Understanding the distribution of property types can inform further analysis, segmentation, or modeling, as different types of properties may have distinct price patterns, features, or market behaviors.

In [ ]:
plt.figure(figsize=(35,10), dpi=200)
ax =sns.countplot(x='listing_agency' , data= df )
plt.xlabel('listing_agency',fontsize=30)
plt.xticks(rotation = 90 , fontsize=15)
plt.ylabel('count',fontsize=30)
plt.yticks(fontsize=20)
plt.title('listing_agency' , fontsize=30)
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(35,10), dpi=200)
ax =sns.countplot(x='city' , data= df )
plt.xlabel('city',fontsize=30)
plt.xticks(rotation = 90 , fontsize=20)
plt.ylabel('count',fontsize=30)
plt.yticks(fontsize=20)
plt.title('city' , fontsize=30)
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(35,10), dpi=200)
ax =sns.countplot(x='zip_code' , data= df )
plt.xlabel('zip_code',fontsize=30)
plt.xticks(rotation = 90 , fontsize=30)
plt.ylabel('count',fontsize=30)
plt.yticks(fontsize=20)
plt.title('zip_code' , fontsize=30)
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(35,10), dpi=200)
ax =sns.countplot(x='product_depth' , data= df )
plt.xlabel('product_depth',fontsize=30)
plt.xticks(rotation = 90 , fontsize=30)
plt.ylabel('count',fontsize=30)
plt.yticks(fontsize=20)
plt.title('product_depth' , fontsize=30)
plt.grid()
plt.show()

- **Premiere**: This category represents the most substantial part of the dataset. These properties might be characterized by high quality, premium features, or exclusive locations.
- **Standard**: The second most common category, "Standard" properties, may include typical residential or commercial offerings with standard features and locations.
- **Basic**: The "Basic" category has a lower count, potentially representing more affordable or entry-level properties.


In [ ]:
plt.figure(figsize=(35,10), dpi=200)
ax =sns.countplot(x='bedroom_count' , data= df )
plt.xlabel('bedroom_count',fontsize=30)
plt.xticks(rotation = 90 , fontsize=30)
plt.ylabel('count',fontsize=30)
plt.yticks(fontsize=20)
plt.title('bedroom_count' , fontsize=30)
plt.grid()
plt.show()

- **3 Bedrooms**: The most common configuration, likely reflecting typical family homes or standard residential units.
- **2 Bedrooms**: The second most frequent count, possibly representing apartments, townhouses, or smaller family homes.
- **4 Bedrooms**: This category might include larger family homes, luxury apartments, or properties with additional rooms for offices or guests.
- **1 Bedroom and Fewer**: These include studio apartments, single-bedroom units, or specialized living arrangements, such as student housing.
- **More than 4 Bedrooms**: These properties may include large family homes, multi-generational living, or high-end luxury properties.

In [ ]:
plt.figure(figsize=(35,10), dpi=200)
ax =sns.countplot(x='bathroom_count' , data= df )
plt.xlabel('bathroom_count',fontsize=30)
plt.xticks(rotation = 90 , fontsize=30)
plt.ylabel('count',fontsize=30)
plt.yticks(fontsize=20)
plt.title('bathroom_count' , fontsize=30)
plt.grid()
plt.show()

- **2 Bathrooms**: The most common configuration, reflecting a standard setup in many family homes, apartments, and townhouses.
- **1 Bathroom**: The second most frequent count, possibly found in smaller apartments, single-bedroom units, or budget-friendly properties.
- **3 Bathrooms**: These may be associated with larger family homes or properties with additional amenities.
- **More than 3 Bathrooms**: Properties with more than three bathrooms are less common and may represent luxury homes, larger family residences, or commercial properties.

In [ ]:
plt.figure(figsize=(35,10), dpi=200)
ax =sns.countplot(x='parking_count' , data= df )
plt.xlabel('parking_count',fontsize=30)
plt.xticks(rotation = 90 , fontsize=30)
plt.ylabel('count',fontsize=30)
plt.yticks(fontsize=20)
plt.title('parking_count' , fontsize=30)
plt.grid()
plt.show()

- **2 Parking Spaces**: The most common configuration, often found in family homes, townhouses, and apartments with dedicated parking.
- **1 Parking Space**: The second most frequent count, reflecting properties with limited parking, such as smaller apartments or urban locations.
- **0 Parking Spaces**: A significant number of properties have no parking spaces, possibly representing inner-city apartments, shared housing, or properties relying on street parking.
- **More than 2 Parking Spaces**: These may include larger family homes, luxury properties, or commercial real estate with additional parking amenities.

In [ ]:
# Select the columns to be plotted
plot_cols = df.columns[ df.columns.isin(['price']) ]

# Calculate the number of rows for subplots
n_cols = 1
n_rows = -(-len(plot_cols) // n_cols)  # ceil division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 5), squeeze=False)



for i, col in enumerate(plot_cols):
    row, col_idx = divmod(i, n_cols)
    sns.histplot(data=df, x=col, kde=True, edgecolor="0", palette='Paired', ax=axes[row, col_idx], multiple='dodge', shrink=0.8)
    axes[row, col_idx].set_title(f'Distribution of {col}')
    axes[row, col_idx].set_xlabel(col)
    axes[row, col_idx].set_ylabel('Frequency')

# Remove empty subplots
for j in range(i+1, n_rows*n_cols):
    row, col_idx = divmod(j, n_cols)
    fig.delaxes(axes[row][col_idx])

plt.tight_layout()
plt.show()

The histogram visualizes the distribution of property prices in the dataset, providing insights into the range, central tendencies, and variations in pricing:

- **Distribution Shape**: The histogram shows a right-skewed distribution, with most properties falling in the lower to middle price range and fewer properties at the higher end.
- **Peak**: The peak of the distribution represents the most common price range, likely reflecting typical residential properties or standard market offerings.
- **Tail**: The long tail extending to the right indicates the presence of higher-priced properties, such as luxury homes, commercial real estate, or exclusive locations.
- **Kernel Density Estimate (KDE)**: The smooth line (KDE) provides an estimate of the underlying probability density function, highlighting the overall shape and patterns in the price distribution.

Understanding the price distribution is essential for market analysis, pricing strategies, investment decisions, and modeling property values. It helps identify opportunities, risks, and trends in the real estate market.

# Conclusion :


### 1. **Property Prices**:
  - **Distribution**: The prices of properties are right-skewed, with a majority in the lower to mid-range and a tail extending to higher-priced luxury or commercial properties.

### 2. **Property Types**:
  - **Houses and Units**: These are the most common types, reflecting the core residential market.
  - **Apartments and Others**: These include specialized or alternative living arrangements.

### 3. **Location Analysis**:
  - **Cities**: A mix of major urban centers, suburban areas, and rural locations was observed. Major cities likely represent key real estate hotspots.
  - **Zip Codes**: There's a wide distribution across zip codes, with dense urban areas, growing suburbs, and less developed regions.

### 4. **Listing Agencies**:
  - **Market Structure**: A combination of major players, moderate agencies, and smaller firms, reflecting a competitive and diverse market.

### 5. **Product Depth**:
  - **Premiere to Basic**: A categorization reflecting quality, features, or market tier. "Premiere" properties were the most common.

### 6. **Property Features**:
  - **Bedroom Count**: Primarily 2-3 bedrooms, with variations representing different living arrangements and property sizes.
  - **Bathroom Count**: Typically 1-2 bathrooms, indicative of standard family homes or apartments.
  - **Parking Count**: Ranging from no parking to multiple spaces, reflecting urban planning, property type, and consumer needs.

### 7. **Inter-relationships and Implications**:
  - **Price & Features**: The price likely correlates with property type, location, and features such as bedrooms, bathrooms, and parking.
  - **Location & Agency**: Different agencies may specialize in certain locations or property types, influencing marketing and investment strategies.
  - **Product Depth & Type**: The categorization into "Premiere," "Standard," and "Basic" may relate to property type, location, and price, guiding segmentation and targeting.

### 8. **Scientific and Strategic Considerations**:
  - **Market Segmentation**: Understanding these patterns enables targeted marketing, investment, and development strategies.
  - **Modeling & Prediction**: The relationships between these variables can be leveraged for predictive modeling, such as price prediction or demand forecasting.
  - **Urban Planning & Policy**: Insights into location, property types, and features can inform urban planning, housing policies, and community development.

### 9. **Limitations & Future Directions**:
  - **Data Quality**: In this part, we were involved in many problems related to data. From important columns that were accompanied by empty cells to location features that could be displayed in a better way. It can be said with confidence that if we had more basic parameters and of course a higher number of data, we could achieve a comprehensive modeling, but with this amount of data and this quality, modeling is a futile task.
  - **Additional Variables**: Exploration of other features, such as amenities, neighborhood characteristics, or historical price trends.
  - **Advanced Analytics**: Application of machine learning or econometric models to derive deeper insights and predictions.
